# Simulacro — Grafos

**Estructuras de Datos**

### Condiciones del simulacro

- El simulacro tiene dos puntos.
- Uno de los puntos debe resolverse usando **matriz de adyacencia** y el otro usando **lista de adyacencia**. Cada estudiante decide qué representación usa en cada punto, pero no puede usar la misma en ambos.
- Las clases base del curso están en la siguiente celda; ejecútela antes de empezar y úselas directamente (`GraphL`, `GraphM`, `GraphWL`, `GraphWM`).
- Cada punto trae al final una celda con casos de prueba. Úselos para construir los grafos y verificar sus resultados.

In [ ]:
# Clases base del curso (copiadas de 08_grafos/base.py)
from typing import Any, Dict, List, Tuple


class GraphL:
    """Grafo (sin peso) representado con lista de adyacencia."""

    def __init__(self) -> None:
        self.adj_list: Dict[Any, List[Any]] = {}
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.adj_list:
            return
        self.adj_list[value] = []
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, directed: bool = True) -> None:
        if vertex_1 not in self.adj_list:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.adj_list:
            self.add_vertex(vertex_2)

        if vertex_2 not in self.adj_list[vertex_1]:
            self.adj_list[vertex_1].append(vertex_2)
        if not directed and vertex_1 not in self.adj_list[vertex_2]:
            self.adj_list[vertex_2].append(vertex_1)

    def __repr__(self) -> str:
        return "\n".join(f"{v}: {ns}" for v, ns in self.adj_list.items())


class GraphM:
    """Grafo (sin peso) representado con matriz de adyacencia."""

    def __init__(self) -> None:
        self.adj_matrix: List[List[int]] = []
        self.nodes: List[Any] = []
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.nodes:
            return
        self.nodes.append(value)
        for row in self.adj_matrix:
            row.append(0)
        self.adj_matrix.append([0] * len(self.nodes))
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, directed: bool = True) -> None:
        if vertex_1 not in self.nodes:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.nodes:
            self.add_vertex(vertex_2)

        pos_v1 = self.nodes.index(vertex_1)
        pos_v2 = self.nodes.index(vertex_2)

        self.adj_matrix[pos_v1][pos_v2] = 1
        if not directed:
            self.adj_matrix[pos_v2][pos_v1] = 1

    def __repr__(self) -> str:
        header = "   " + " ".join(str(n) for n in self.nodes)
        rows = [
            str(self.nodes[i]) + "  " + " ".join(map(str, row))
            for i, row in enumerate(self.adj_matrix)
        ]
        return "\n".join([header, *rows])


class GraphWL:
    """Grafo ponderado representado con lista de adyacencia."""

    def __init__(self) -> None:
        self.adj_list: Dict[Any, List[Tuple[Any, float]]] = {}
        self.size: int = 0

    def add_vertex(self, value: Any) -> None:
        if value in self.adj_list:
            return
        self.adj_list[value] = []
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, weight: float, directed: bool = True) -> None:
        if vertex_1 not in self.adj_list:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.adj_list:
            self.add_vertex(vertex_2)

        self.adj_list[vertex_1].append((vertex_2, weight))
        if not directed:
            self.adj_list[vertex_2].append((vertex_1, weight))

    def __repr__(self) -> str:
        return "\n".join(f"{v}: {ns}" for v, ns in self.adj_list.items())


class GraphWM:
    """Grafo ponderado representado con matriz de adyacencia."""

    def __init__(self, non_representative_value: float = 0) -> None:
        self.adj_matrix: List[List[float]] = []
        self.nodes: List[Any] = []
        self.size: int = 0
        self.non_representative_value = non_representative_value

    def add_vertex(self, value: Any) -> None:
        if value in self.nodes:
            return
        self.nodes.append(value)
        for row in self.adj_matrix:
            row.append(self.non_representative_value)
        self.adj_matrix.append([self.non_representative_value] * len(self.nodes))
        self.size += 1

    def add_edge(self, vertex_1: Any, vertex_2: Any, weight: float, directed: bool = True) -> None:
        if vertex_1 not in self.nodes:
            self.add_vertex(vertex_1)
        if vertex_2 not in self.nodes:
            self.add_vertex(vertex_2)

        pos_v1 = self.nodes.index(vertex_1)
        pos_v2 = self.nodes.index(vertex_2)

        self.adj_matrix[pos_v1][pos_v2] = weight
        if not directed:
            self.adj_matrix[pos_v2][pos_v1] = weight

    def __repr__(self) -> str:
        header = "   " + " ".join(str(n) for n in self.nodes)
        rows = [
            str(self.nodes[i]) + "  " + " ".join(map(str, row))
            for i, row in enumerate(self.adj_matrix)
        ]
        return "\n".join([header, *rows])

## Punto 1 — Envíos con presupuesto exacto

Una empresa de mensajería tiene convenios con varias bodegas. Cada trayecto directo entre dos bodegas tiene un costo de peaje, y los trayectos son de un solo sentido: poder ir de A a B no implica poder ir de B a A, y si ambos existen pueden costar distinto.

Para una campaña de fin de año, la empresa le asigna a cada repartidor un bono de peajes de valor fijo K, que solo se puede usar si se gasta **completo**. Se quiere conocer todas las rutas posibles en las que el bono se gasta exactamente.

Dado un grafo dirigido y ponderado G, donde los nodos son bodegas y el peso de cada arista es el costo del trayecto, y un entero K, devuelva **todas las rutas cuyo costo acumulado sea exactamente K**.

**Entrada**
- El grafo G, descrito por su lista de nodos y su lista de aristas `(origen, destino, peso)`.
- Un entero K.

**Salida**
- Una lista con todas las rutas que cumplen la condición. Cada ruta es la lista de nodos en el orden en que se recorren. Si no hay ninguna, se devuelve una lista vacía.

**Condiciones**
- Una ruta puede empezar en cualquier nodo del grafo y debe tener al menos una arista.
- Una ruta no puede pasar dos veces por el mismo nodo.
- Todos los pesos son enteros positivos y K ≥ 1.
- El orden en que se devuelven las rutas no importa; el orden de los nodos dentro de cada ruta sí.

In [ ]:
casos_ej_018 = [
    {
        "entrada": {
            "grafo": {
                "nodos": ["A", "B", "C", "D", "E"],
                "aristas": [("A", "B", 2), ("A", "C", 4), ("B", "C", 1), ("B", "D", 5),
                            ("C", "D", 3), ("D", "A", 1), ("C", "E", 2), ("E", "D", 1)],
            },
            "K": 6,
        },
        "salida": [
            ["A", "B", "C", "D"], ["A", "B", "C", "E", "D"], ["A", "C", "E"],
            ["B", "D", "A"], ["C", "D", "A", "B"], ["C", "E", "D", "A", "B"],
            ["D", "A", "B", "C", "E"], ["E", "D", "A", "C"],
        ],
    },
    {
        "entrada": {
            "grafo": {
                "nodos": ["P", "Q", "R", "S"],
                "aristas": [("P", "Q", 1), ("Q", "R", 1), ("R", "S", 1), ("P", "S", 3),
                            ("Q", "S", 2), ("S", "P", 2)],
            },
            "K": 3,
        },
        "salida": [
            ["P", "Q", "R", "S"], ["P", "Q", "S"], ["P", "S"], ["R", "S", "P"], ["S", "P", "Q"],
        ],
    },
    {
        "entrada": {
            "grafo": {"nodos": ["A", "B", "C"], "aristas": [("A", "B", 3), ("B", "C", 3), ("C", "A", 3)]},
            "K": 6,
        },
        "salida": [["A", "B", "C"], ["B", "C", "A"], ["C", "A", "B"]],
    },
    {
        "entrada": {
            "grafo": {"nodos": ["A", "B", "C"], "aristas": [("A", "B", 3), ("B", "C", 3), ("C", "A", 3)]},
            "K": 9,
        },
        "salida": [],
    },
    {
        "entrada": {
            "grafo": {"nodos": ["A", "B", "C"], "aristas": [("A", "B", 2), ("B", "C", 2)]},
            "K": 5,
        },
        "salida": [],
    },
    {
        "entrada": {"grafo": {"nodos": ["A"], "aristas": []}, "K": 3},
        "salida": [],
    },
]

## Punto 2 — Cierre de estaciones

Una red de distribución de agua está formada por estaciones conectadas por tuberías de un solo sentido. Tras un problema de contaminación en una estación N, la empresa decide cerrar preventivamente **todas las estaciones que estén a distancia exactamente K de N**, y retirar del mapa las tuberías que entran o salen de ellas.

La distancia de N a una estación es el menor número de tuberías que hay que recorrer, respetando su sentido, para llegar desde N hasta ella. Una estación a la que no se puede llegar desde N no está a ninguna distancia de N.

Dado un grafo dirigido y no ponderado G, un nodo N y un entero K, elimine del grafo todos los nodos que estén a distancia exactamente K de N, junto con sus aristas, y devuelva el grafo resultante.

**Entrada**
- El grafo G, descrito por su lista de nodos y su lista de aristas `(origen, destino)`.
- El nodo N, que siempre pertenece al grafo.
- Un entero K ≥ 1.

**Salida**
- El grafo resultante, descrito por su lista de nodos y su lista de aristas.

**Condiciones**
- El grafo puede tener ciclos.
- En la salida, el orden de los nodos y de las aristas no importa.

In [ ]:
casos_ej_019 = [
    {
        "entrada": {
            "grafo": {
                "nodos": ["A", "B", "C", "D", "E", "F"],
                "aristas": [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E"), ("E", "F")],
            },
            "N": "A",
            "K": 2,
        },
        "salida": {
            "nodos": ["A", "B", "C", "E", "F"],
            "aristas": [("A", "B"), ("A", "C"), ("E", "F")],
        },
    },
    {
        "entrada": {
            "grafo": {
                "nodos": ["A", "B", "C", "D", "E"],
                "aristas": [("A", "B"), ("B", "C"), ("C", "D"), ("A", "D"), ("D", "E")],
            },
            "N": "A",
            "K": 2,
        },
        "salida": {
            "nodos": ["A", "B", "D"],
            "aristas": [("A", "B"), ("A", "D")],
        },
    },
    {
        "entrada": {
            "grafo": {
                "nodos": ["A", "B", "C", "D", "E"],
                "aristas": [("A", "B"), ("B", "C"), ("C", "D"), ("A", "D"), ("D", "E")],
            },
            "N": "A",
            "K": 3,
        },
        "salida": {
            "nodos": ["A", "B", "C", "D", "E"],
            "aristas": [("A", "B"), ("B", "C"), ("C", "D"), ("A", "D"), ("D", "E")],
        },
    },
    {
        "entrada": {
            "grafo": {
                "nodos": ["A", "B", "C", "D"],
                "aristas": [("A", "B"), ("B", "C"), ("C", "A"), ("D", "A")],
            },
            "N": "A",
            "K": 1,
        },
        "salida": {
            "nodos": ["A", "C", "D"],
            "aristas": [("C", "A"), ("D", "A")],
        },
    },
    {
        "entrada": {
            "grafo": {
                "nodos": ["1", "2", "3", "4", "5"],
                "aristas": [("1", "2"), ("2", "3"), ("3", "1"), ("3", "4"), ("5", "3")],
            },
            "N": "1",
            "K": 2,
        },
        "salida": {
            "nodos": ["1", "2", "4", "5"],
            "aristas": [("1", "2")],
        },
    },
    {
        "entrada": {
            "grafo": {"nodos": ["A", "B", "C"], "aristas": [("A", "B"), ("B", "C")]},
            "N": "A",
            "K": 5,
        },
        "salida": {"nodos": ["A", "B", "C"], "aristas": [("A", "B"), ("B", "C")]},
    },
    {
        "entrada": {"grafo": {"nodos": ["X"], "aristas": []}, "N": "X", "K": 1},
        "salida": {"nodos": ["X"], "aristas": []},
    },
]